<a href="https://colab.research.google.com/github/mohamedYehiamahmoud/Data-Analyst-Agent/blob/main/AutoAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
#!pip install langgraph langchain langchain-groq pandas matplotlib seaborn python-dotenv langchain-experimental

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import os
import pandas as pd
from typing import Literal, Annotated, TypedDict, Optional, List
from langgraph.graph import StateGraph, END, START
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from langchain_experimental.tools.python.tool import PythonAstREPLTool
import operator
from langchain_groq import ChatGroq
from google.colab import userdata

In [24]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
GROQ_MODEL = "llama-3.1-8b-instant"

In [25]:
REPHRASED_QUERY_PROMPT = """
Rephrase the user query to be more specific and analytical for pandas DataFrame operations.

Original Query: {query}
Available Columns: {df_columns}

Rephrased Query (focus on actionable data analysis):
"""

RELEVANCY_CHECK_PROMPT = """
You are a data analysis assistant. Your job is to check if the user query is relevant to the available dataframe columns.

Available Columns:
{df_columns}

User Query: {query}

Answer ONLY 'yes' if the query can be answered using the available columns, or 'no' if it cannot.
"""

sys_msg = """
You are an expert data analyst assistant.
Analyze the query against the provided dataframe columns and generate accurate Python code using pandas.

DataFrame Columns Description:
{df_columns}

DataFrame Sample:
{df_head}

Guidelines:
- Always use safe pandas operations
- Handle missing values appropriately
- Generate clear, commented code
- For visualizations: use matplotlib/seaborn, save to 'images/' folder with uuid filename
- Return reports in clean markdown format
"""

user_msg = """
Generate a comprehensive markdown report based on the analysis results.

Original Query: {query}
Analysis Results: {execution_results}
DataFrame Context: {df_columns}

Report Requirements:
- Executive summary
- Key findings with numerical insights
- Reference to any generated charts (with relative image paths)
- Actionable recommendations
- Format in clean markdown (no ```markdown wrapper)
"""

In [26]:
class AgentState(TypedDict):
    """State schema for the LangGraph agent."""
    query: str
    csv_file_path: str
    column_description: str
    rephrased_query: Optional[str]
    Python_Code: Optional[str]
    data_frame: Optional[pd.DataFrame]
    execution_results: Optional[str]
    execution_error: Optional[str]
    reports: Optional[str]
    Python_script_check: int
    max_Python_script_check: int
    script_security_issues: Optional[str]
    next_node: Optional[str]
    is_safe: Optional[bool]

In [27]:
def check_query_relevancy(state: AgentState) -> AgentState:
    print("--- QUERY RELEVANCY CHECK ---")

    query = state["query"]
    column_description = state["column_description"]

    class grade(BaseModel):
        """Binary score for relevance check."""
        binary_score: str = Field(description="Relevance score 'yes' or 'no'")

    model = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    llm_with_tool = model.with_structured_output(grade)

    prompt = PromptTemplate(
        template=RELEVANCY_CHECK_PROMPT,
        input_variables=["df_columns", "query"],
    )

    chain = prompt | llm_with_tool
    scored_result = chain.invoke({"df_columns": column_description, "query": query})
    grade_result = scored_result.binary_score

    if grade_result == "yes":
        return {"next_node": "re_write_query"}
    else:
        return {"next_node": "query_relevancy_report"}

In [28]:
def re_write_query(state: AgentState) -> AgentState:
    print("--- Re WRITING QUERY ---")

    query = state["query"]
    column_description = state["column_description"]

    rephrased_query_prompt = PromptTemplate(
        template=REPHRASED_QUERY_PROMPT,
        input_variables=["query", "df_columns"],
    )

    model = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    rephrased_chain = rephrased_query_prompt | model | StrOutputParser()
    rephrased_question = rephrased_chain.invoke({"query": query, "df_columns": column_description})

    print(f"Rephrased Query: {rephrased_question}")
    return {"rephrased_query": rephrased_question}

In [29]:
def generate_Python_code(state: AgentState) -> AgentState:
    print("--- PYTHON CODE GENERATOR ---")

    rephrased_query = state["rephrased_query"]
    rephrased_query = f"""
    {rephrased_query}
    Do it in Python and generate a single report.
    Combine numerical analysis with observations, visualizations and charts.
    If any visualization is needed, save it in images folder with unique file name including uuid.
    Include the saved image with its RELATIVE PATH in the reports as markdown.
    Generate the reports in markdown format (DO NOT INCLUDE ```markdown) and print the markdown reports.
    """

    csv_file_path = state["csv_file_path"]
    column_description = state["column_description"]
    df = pd.read_csv(csv_file_path)

    df_head = str(df.sample(min(10, len(df))).to_markdown())

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(sys_msg),
        HumanMessagePromptTemplate.from_template("{rephrased_query}"),
    ])

    llm = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    chain = prompt | llm | StrOutputParser()

    code = chain.invoke({
        "df_head": df_head,
        "df_columns": column_description,
        "rephrased_query": rephrased_query
    })

    return {
        "rephrased_query": rephrased_query,
        "Python_Code": code,
        "data_frame": df,
    }

In [30]:
class SanitizingScript(BaseModel):
    is_safe: bool = Field(description="Is the Python script safe to execute?")
    reason: str = Field(description="Explanation if the script is not safe")

def sanitize_python_script(state: AgentState) -> AgentState:
    print("--- SANITIZE PYTHON SCRIPT ---")

    python_script = state['Python_Code']

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a Python security expert.
        Check if the following Python script is safe to execute.
        Look for: file deletion, system calls, network calls, infinite loops, or any destructive operations."""),
        HumanMessage(content=f"Python script: {python_script}")
    ])

    llm = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    sanitize_chain = prompt | llm.with_structured_output(
        schema=SanitizingScript,
        method="function_calling",
        include_raw=False,
    )

    response = sanitize_chain.invoke({'input': ''})

    if response.is_safe:
        return {"is_safe": True, "script_security_issues": None}
    else:
        return {
            "is_safe": False,
            "script_security_issues": response.reason,
        }

def route_after_sanitize(state: AgentState) -> str:
    Python_script_check = state['Python_script_check']
    max_Python_script_check = state['max_Python_script_check']

    if state["is_safe"]:
        return "execute_python_code"
    else:
        if Python_script_check >= max_Python_script_check:
            return END
        return "re_generate_python_code"

In [31]:
def make_decision(state: AgentState) -> str:
    print("--- MAKING DECISION ---")

    execution_error = state.get("execution_error", None)
    Python_script_check = state['Python_script_check']
    max_Python_script_check = state['max_Python_script_check']

    if execution_error:
        if Python_script_check >= max_Python_script_check:
            return END
        return "re_generate_python_code"
    else:
        return "generate_report"

In [32]:
def run_python_code(state: AgentState) -> AgentState:
    print("--- PYTHON CODE EXECUTER ---")

    Python_script = state['Python_Code']
    df = state["data_frame"]
    df_locals = {"df": df}
    python_repl = PythonAstREPLTool(locals=df_locals)

    os.makedirs("images", exist_ok=True)

    try:
        results = python_repl.run(Python_script)
        if "error" in results.lower():
            return {"execution_error": results, "execution_results": None}
        else:
            return {"execution_results": results, "execution_error": None}
    except Exception as e:
        return {"execution_error": str(e), "execution_results": None}

In [33]:
def generate_report(state: AgentState) -> AgentState:
    print("--- REPORT GENERATOR ---")

    query = state["query"]
    column_description = state["column_description"]
    execution_results = state["execution_results"]

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(sys_msg),
        HumanMessagePromptTemplate.from_template(user_msg),
    ])

    llm = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    chain = prompt | llm | StrOutputParser()

    df = state["data_frame"]
    df_head = str(df.sample(min(10, len(df))).to_markdown())

    reports = chain.invoke({
        "query": query,
        "execution_results": execution_results,
        "df_columns": column_description,
        "df_head": df_head
    })

    return {"reports": reports}

In [34]:
def go_to_next(state: AgentState) -> str:
    return state['next_node']

def query_relevancy_report(state: AgentState) -> dict:
    print("--- QUERY NOT RELEVANT REPORT ---")
    return {
        "reports": f"## Query Not Relevant\n\nThe query '{state['query']}' cannot be answered with the available columns:\n\n{state['column_description']}\n\nPlease rephrase your question to reference the available data fields."
    }

In [35]:
def re_generate_Python_code(state: AgentState) -> AgentState:
    print("--- RE-GENERATING PYTHON CODE (WITH ERROR CONTEXT) ---")

    error_msg = state.get("execution_error") or state.get("script_security_issues", "Unknown error")
    previous_code = state.get("Python_Code", "")

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a Python and pandas expert. Your task is to fix previously failed code.
         Stick to pandas only, handle missing values appropriately, and save any visualizations to the 'images/' directory.
         Output ONLY the fixed Python code. Do not explain, do not add markdown fences."""),
        HumanMessage(content=f"""
         The previous code failed due to: {error_msg}

         Previous code:
         {previous_code}

         Rewrite the complete code to specifically resolve this issue.
         """)
    ])

    llm = ChatGroq(api_key=userdata.get("GROQ_API_KEY"), temperature=0, model=GROQ_MODEL)
    new_code = (prompt | llm | StrOutputParser()).invoke({})

    return {
        "Python_Code": new_code,
        "execution_error": None,
        "script_security_issues": None,
        "is_safe": None,
        "Python_script_check": state["Python_script_check"] + 1
    }

In [36]:
workflow = StateGraph(AgentState)

# Nodes
workflow.add_node("check_query_relevancy", check_query_relevancy)
workflow.add_node("query_relevancy_report", query_relevancy_report)
workflow.add_node("re_write_query", re_write_query)
workflow.add_node("generate_python_code", generate_Python_code)
workflow.add_node("sanitize_python_script", sanitize_python_script)
workflow.add_node("execute_python_code", run_python_code)
workflow.add_node("re_generate_python_code", re_generate_Python_code)
workflow.add_node("generate_report", generate_report)

# Edges
workflow.add_edge(START, "check_query_relevancy")
workflow.add_conditional_edges("check_query_relevancy", go_to_next)
workflow.add_edge("re_write_query", "generate_python_code")

workflow.add_edge("generate_python_code", "sanitize_python_script")

workflow.add_conditional_edges("sanitize_python_script", route_after_sanitize)

workflow.add_conditional_edges("execute_python_code", make_decision)
workflow.add_edge("re_generate_python_code", "sanitize_python_script")
workflow.add_edge("query_relevancy_report", END)
workflow.add_edge("generate_report", END)

graph = workflow.compile()

try:
    graph.get_graph(xray=1).draw_mermaid_png(output_file_path="pandas_dataframe_qa.png")
except Exception as e:
    print(f"Failed to generate graph image: {e}")

In [37]:
def generate_column_description(csv_path):
    df = pd.read_csv(csv_path)
    desc = []

    for col in df.columns:
        dtype = str(df[col].dtype)
        unique_count = df[col].nunique()

        if dtype in ['int64', 'float64']:
            col_type = f"numeric ({dtype})"
            stats = f"range: {df[col].min()} - {df[col].max()}"
        elif unique_count < 10:
            col_type = "categorical"
            stats = f"unique values: {df[col].unique().tolist()}"
        else:
            col_type = "text"
            stats = f"unique count: {unique_count}"

        desc.append(f"- {col}: {col_type}, {stats}")

    return "\n".join(desc)

In [38]:
user_query = 'What is the mode of gender in the dataset?'
csv_file_path = f'/content/drive/MyDrive/datasets/tips.csv'

column_description = generate_column_description(csv_file_path)
print(column_description)

- total_bill: numeric (float64), range: 3.07 - 50.81
- tip: numeric (float64), range: 1.0 - 10.0
- sex: categorical, unique values: ['Female', 'Male']
- smoker: categorical, unique values: ['No', 'Yes']
- day: categorical, unique values: ['Sun', 'Sat', 'Thur', 'Fri']
- time: categorical, unique values: ['Dinner', 'Lunch']
- size: numeric (int64), range: 1 - 6


In [39]:
results = graph.invoke({
    "query": user_query,
    "csv_file_path": csv_file_path,
    "column_description": column_description,
    "Python_script_check": 0,
    "max_Python_script_check": 5
})

print(results.get("reports", "No report generated."))

--- QUERY RELEVANCY CHECK ---
--- Re WRITING QUERY ---
Rephrased Query: To determine the mode of the 'sex' column in the dataset, we can use the `mode()` function provided by pandas. Here's a more specific and analytical approach:

1. **Check for missing values**: Before calculating the mode, ensure there are no missing values in the 'sex' column using `df['sex'].isnull().sum()`.
2. **Calculate the mode**: Use `df['sex'].mode()` to find the most frequently occurring value in the 'sex' column.
3. **Verify the result**: If there are multiple modes, use `df['sex'].value_counts()` to get a count of each unique value and verify the result.

Here's the code:
```python
import pandas as pd

# Check for missing values
missing_values = df['sex'].isnull().sum()
print(f"Missing values in 'sex' column: {missing_values}")

# Calculate the mode
mode_sex = df['sex'].mode()
print(f"Mode of 'sex' column: {mode_sex}")

# Verify the result
value_counts = df['sex'].value_counts()
print(f"Value counts of 's